# 读取数据文件

In [34]:
# import pandas as pd
# import numpy as np
# #climate data
# #From GitHub
# climate_table = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\civ-rainfall-subnat-5ytd.csv')

# #2022 education data
# CP_2022 = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\Dictee_jour_CP_2022_2023.xlsx', sheet_name=0)
# CE_CM_2022 = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\CE_CM_DICTEE_JOUR_2022_2023.xlsx', sheet_name=0)

# #2023 education data
# CP_2023 = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\DICTEE_CP_2023_2024.xls', sheet_name=0)
# CE_CM_2023 = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\DICTEE CE_CM_2023_2024.xls', sheet_name=0)

# #2024 education data
# CP_2024_LEXI = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\LEXICAL_CP_2024_2025_20250723_112750.xlsx', sheet_name=0)
# CE_CM_2024_CONJ = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\CONJUGAISON_CE_CM_2024_2025_20250723_113311.xlsx', sheet_name=0)
# CE_CM_2024_GRAM = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\GRAMMAIRE_CE_CM_2024_2025_20250723_113109.xlsx', sheet_name=0)
# CE_CM_2024_LEXI = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\LEXICAL_CE_CM_2024_2025_20250723_112836.xlsx', sheet_name=0)


In [35]:
##导入表
import pandas as pd
import numpy as np

# #2022 education data
# CP_2022 = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\Dictee_jour_CP_2022_2023.xlsx', sheet_name=0)
# CE_CM_2022 = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\CE_CM_DICTEE_JOUR_2022_2023.xlsx', sheet_name=0)

# #2023 education data
# CP_2023 = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\DICTEE_CP_2023_2024.xls', sheet_name=0)
# CE_CM_2023 = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\DICTEE CE_CM_2023_2024.xls', sheet_name=0)


# 处理气候数据

In [36]:
climate_table = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\civ-rainfall-subnat-full.csv')
climate_table['date'] = pd.to_datetime(climate_table['date'])
climate_table['month'] = climate_table['date'].dt.month
climate_table['year'] = climate_table['date'].dt.year


##  新增变量

# 月间波动
climate_table['FluctMonth'] = (climate_table.groupby('PCODE')['r1h'].diff(3).abs())/(climate_table['r1h'])

# 月内波动 ---
# 计算 |rfh_i - rfh_{i-1}|
diff_rfh = climate_table.groupby('PCODE')['rfh'].diff().abs()
# 计算 (|rfh_i - rfh_{i-1}| + |rfh_{i-1} - rfh_{i-2}|) / 2 / r1h_i  用 shift(1) 获取上一行的差分值
climate_table['FluctIntra'] = ((diff_rfh + diff_rfh.shift(1)) / 2 )/ climate_table['r1h']

# 旬间波动
climate_table['FluctTenDays'] = (climate_table.groupby('PCODE')['rfh'].diff().abs())/(climate_table['rfh'])#.shift(1)

# 还有r1h,rfh,r3h
# 'rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays',




# 30日降雨量历史波动 log(r1h / r1h_avg)
climate_table['rain_hist_vol'] = np.log((climate_table['r1h'] + 1e-6) / (climate_table['r1h_avg'] + 1e-6))

# 确保数据按 PCODE 和日期排序，以便进行行差分计算
climate_table = climate_table.sort_values(['PCODE', 'date']).reset_index(drop=True)

# 3. Heavy_Rain & Low_Rain
# 按 PCODE 和月份分组，计算 r1h 的分位数
g_r1h = climate_table.groupby(['PCODE', 'month'])['r1h']
q90 = g_r1h.transform(lambda x: x.quantile(0.8))##可更改@@@@@@@
q10 = g_r1h.transform(lambda x: x.quantile(0.2))##可更改@@@@@@@

climate_table['Heavy_Rain'] = (climate_table['r1h'] > q90).astype(int)
climate_table['Low_Rain'] = (climate_table['r1h'] < q10).astype(int)

# # 计算 r1h 在 PCODE 和 month 分组内的百分位排名 (0-1)
# climate_table['Heavy_Rain'] = climate_table.groupby(['PCODE', 'month'])['r1h'].rank(pct=True)


# 4. 月间极端波动 ---
# 计算第 i 行与 i-3 行的绝对差
climate_table['diff_m'] = climate_table.groupby('PCODE')['r1h'].diff(3).abs()  #这里更改了，除了一个当期值￥￥￥￥
# climate_table['diff_m'] = (climate_table.groupby('PCODE')['rfh'].diff().abs())/climate_table['rfh']

# 过滤掉 1981 年的数据进行统计计算
climate_table_filtered = climate_table[climate_table['year'] != 1981].copy()
g_diff_m = climate_table_filtered.groupby(['PCODE', 'month'])['diff_m']
q90_m = g_diff_m.transform(lambda x: x.quantile(0.8))##可更改@@@@@@@
climate_table.loc[climate_table['year'] != 1981, 'Extreme_Fluct_Month'] = (climate_table_filtered['diff_m'] > q90_m).astype(int)



# 5. 月内极端波动 ---
# 计算 |rfh_i - rfh_{i-1}|
diff_rfh = climate_table.groupby('PCODE')['rfh'].diff().abs()
# 计算 (|rfh_i - rfh_{i-1}| + |rfh_{i-1} - rfh_{i-2}|) / 2 / r1h_i
# 用 shift(1) 获取上一行的差分值
climate_table['intra_val'] = ((diff_rfh + diff_rfh.shift(1)) / 2 )/ climate_table['r1h']

# 过滤掉 1981 年的数据进行统计计算
climate_table_filtered_intra = climate_table[climate_table['year'] != 1981].copy()
g_intra = climate_table_filtered_intra.groupby(['PCODE', 'month'])['intra_val']
q90_intra = g_intra.transform(lambda x: x.quantile(0.9))
climate_table.loc[climate_table['year'] != 1981, 'Extreme_Fluct_Intra'] = (climate_table_filtered_intra['intra_val'] > q90_intra).astype(int)


# 填充空值为0 (例如1981年的行或因diff产生的NaN)
climate_table[['Extreme_Fluct_Month', 'Extreme_Fluct_Intra']] = climate_table[['Extreme_Fluct_Month', 'Extreme_Fluct_Intra']].fillna(0).astype(int)
climate_table.to_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\climate_table_for_check.xlsx')
#climate_table表check过，新加的字段都没问题

In [37]:
civ_adminboundaries_tabulardata = pd.read_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\civ_adminboundaries_tabulardata.xlsx', sheet_name=2)
# 使用 merge 进行连接，将climate_table中的PCODE映射到地点方便后继与教育数据聚合
merged_climate_table = pd.merge(
    climate_table.loc[climate_table['adm_level']==2,['date', 'PCODE', 'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays','rain_hist_vol','Heavy_Rain','Low_Rain',\
                                                     'Extreme_Fluct_Month','Extreme_Fluct_Intra']],
    civ_adminboundaries_tabulardata[['ADM2_PCODE', 'ADM2_FR', 'ADM2_REF', 'ADM1_FR', 'ADM1_PCODE']],
    left_on='PCODE', 
    right_on='ADM2_PCODE',
    how='left'
)
merged_climate_table = merged_climate_table[['date', 'PCODE', 'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays','rain_hist_vol','Heavy_Rain','Low_Rain',\
                                             'Extreme_Fluct_Month','Extreme_Fluct_Intra',\
                                             'ADM2_FR', 'ADM2_REF', 'ADM1_FR', 'ADM1_PCODE']]

merged_climate_table.to_excel(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\merged_climate_table.xlsx')

# 年级、听写次数到具体日期的映射表

In [38]:

date_mapping_2022 = {
    # (Clas, SESSION) : 'YYYY/MM/DD'
    ('CP1', 1): '2022/11/17',
    ('CP2', 1): '2022/11/17',
    ('CE1', 1): '2022/11/18',
    ('CE2', 1): '2022/11/18',
    ('CM1', 1): '2022/11/18',
    ('CM2', 1): '2022/11/18',

    ('CP1', 2): '2022/12/1',
    ('CP2', 2): '2022/12/1',
    ('CE1', 2): '2022/12/2',
    ('CE2', 2): '2022/12/2',
    ('CM1', 2): '2022/12/2',
    ('CM2', 2): '2022/12/2',

    ('CP1', 3): '2022/12/15',
    ('CP2', 3): '2022/12/15',
    ('CE1', 3): '2022/12/16',
    ('CE2', 3): '2022/12/16',
    ('CM1', 3): '2022/12/16',
    ('CM2', 3): '2022/12/16',

    ('CP1', 4): '2023/1/19',
    ('CP2', 4): '2023/1/19',
    ('CE1', 4): '2023/1/20',
    ('CE2', 4): '2023/1/20',
    ('CM1', 4): '2023/1/20',
    ('CM2', 4): '2023/1/20',

    ('CP1', 5): '2023/2/2',
    ('CP2', 5): '2023/2/2',
    ('CE1', 5): '2023/2/3',
    ('CE2', 5): '2023/2/3',
    ('CM1', 5): '2023/2/3',
    ('CM2', 5): '2023/2/3',

    ('CP1', 6): '2023/2/23',
    ('CP2', 6): '2023/2/23',
    ('CE1', 6): '2023/2/24',
    ('CE2', 6): '2023/2/24',
    ('CM1', 6): '2023/2/24',
    ('CM2', 6): '2023/2/24',

    ('CP1', 7): '2023/3/9',
    ('CP2', 7): '2023/3/9',
    ('CE1', 7): '2023/3/10',
    ('CE2', 7): '2023/3/10',
    ('CM1', 7): '2023/3/10',
    ('CM2', 7): '2023/3/10',

    ('CP1', 8): '2023/3/30',
    ('CP2', 8): '2023/3/30',
    ('CE1', 8): '2023/3/31',
    ('CE2', 8): '2023/3/31',
    ('CM1', 8): '2023/3/31',
    ('CM2', 8): '2023/3/31',

    ('CP1', 9): '2023/5/4',
    ('CP2', 9): '2023/5/4',
    ('CE1', 9): '2023/5/5',
    ('CE2', 9): '2023/5/5',
    ('CM1', 9): '2023/5/5',
    ('CM2', 9): '2023/5/5',
}

date_mapping_2023 = {
    # (Clas, SESSION) : 'YYYY/MM/DD'
    ('CP1', 1): '2023/11/16',
    ('CP2', 1): '2023/11/16',
    ('CE1', 1): '2023/11/17',
    ('CE2', 1): '2023/11/17',
    ('CM1', 1): '2023/11/17',
    ('CM2', 1): '2023/11/17',

    ('CP1', 2): '2023/12/14',
    ('CP2', 2): '2023/12/14',
    ('CE1', 2): '2023/12/15',
    ('CE2', 2): '2023/12/15',
    ('CM1', 2): '2023/12/15',
    ('CM2', 2): '2023/12/15',

    ('CP1', 3): '2024/1/18',
    ('CP2', 3): '2024/1/18',
    ('CE1', 3): '2024/1/19',
    ('CE2', 3): '2024/1/19',
    ('CM1', 3): '2024/1/19',
    ('CM2', 3): '2024/1/19',

    ('CP1', 4): '2024/2/15',
    ('CP2', 4): '2024/2/15',
    ('CE1', 4): '2024/2/16',
    ('CE2', 4): '2024/2/16',
    ('CM1', 4): '2024/2/16',
    ('CM2', 4): '2024/2/16',

    ('CP1', 5): '2024/3/21',
    ('CP2', 5): '2024/3/21',
    ('CE1', 5): '2024/3/22',
    ('CE2', 5): '2024/3/22',
    ('CM1', 5): '2024/3/22',
    ('CM2', 5): '2024/3/22',

    ('CP1', 6): '2024/4/11',
    ('CP2', 6): '2024/4/11',
    ('CE1', 6): '2024/4/12',
    ('CE2', 6): '2024/4/12',
    ('CM1', 6): '2024/4/12',
    ('CM2', 6): '2024/4/12',

    ('CP1', 7): '2024/4/25',
    ('CP2', 7): '2024/4/25',
    ('CE1', 7): '2024/4/26',
    ('CE2', 7): '2024/4/26',
    ('CM1', 7): '2024/4/26',
    ('CM2', 7): '2024/4/26',
}

date_mapping_2024 = {
    # (Clas, SESSION) : 'YYYY/MM/DD'
    ('CP1', 'NOV 2024'): '2024/11/21',
    ('CP2', 'NOV 2024'): '2024/11/21',
    ('CE1', 'NOV 2024'): '2024/11/22',
    ('CE2', 'NOV 2024'): '2024/11/22',
    ('CM1', 'NOV 2024'): '2024/11/22',
    ('CM2', 'NOV 2024'): '2024/11/22',

    ('CP1', 'DEC 2024'): '2024/12/5',
    ('CP2', 'DEC 2024'): '2024/12/5',
    ('CE1', 'DEC 2024'): '2024/12/6',
    ('CE2', 'DEC 2024'): '2024/12/6',
    ('CM1', 'DEC 2024'): '2024/12/6',
    ('CM2', 'DEC 2024'): '2024/12/6',

    ('CP1', 'JANVIER 2025'): '2025/1/23',
    ('CP2', 'JANVIER 2025'): '2025/1/23',
    ('CE1', 'JANVIER 2025'): '2025/1/24',
    ('CE2', 'JANVIER 2025'): '2025/1/24',
    ('CM1', 'JANVIER 2025'): '2025/1/24',
    ('CM2', 'JANVIER 2025'): '2025/1/24',

    ('CP1', 'MARS 2025'): '2025/3/20',
    ('CP2', 'MARS 2025'): '2025/3/20',
    ('CE1', 'MARS 2025'): '2025/3/21',
    ('CE2', 'MARS 2025'): '2025/3/21',
    ('CM1', 'MARS 2025'): '2025/3/21',
    ('CM2', 'MARS 2025'): '2025/3/21',

    ('CP1', 'MARS  2025'): '2025/3/20',
    ('CP2', 'MARS  2025'): '2025/3/20',
    ('CE1', 'MARS  2025'): '2025/3/21',
    ('CE2', 'MARS  2025'): '2025/3/21',
    ('CM1', 'MARS  2025'): '2025/3/21',
    ('CM2', 'MARS  2025'): '2025/3/21',

    ('CP1', 'AVRIL 2025'): '2025/4/10',
    ('CP2', 'AVRIL 2025'): '2025/4/10',
    ('CE1', 'AVRIL 2025'): '2025/4/11',
    ('CE2', 'AVRIL 2025'): '2025/4/11',
    ('CM1', 'AVRIL 2025'): '2025/4/11',
    ('CM2', 'AVRIL 2025'): '2025/4/11',

    ('CP1', 'AVRIL-MAI 2025'): '2025/5/15',
    ('CP2', 'AVRIL-MAI 2025'): '2025/5/15',
    ('CE1', 'AVRIL-MAI 2025'): '2025/5/16',
    ('CE2', 'AVRIL-MAI 2025'): '2025/5/16',
    ('CM1', 'AVRIL-MAI 2025'): '2025/5/16',
    ('CM2', 'AVRIL-MAI 2025'): '2025/5/16',

    ('CP1', 'JUIN 2025'): '2025/6/15',
    ('CP2', 'JUIN 2025'): '2025/6/15',
    ('CE1', 'JUIN 2025'): '2025/6/16',
    ('CE2', 'JUIN 2025'): '2025/6/16',
    ('CM1', 'JUIN 2025'): '2025/6/16',
    ('CM2', 'JUIN 2025'): '2025/6/16',
}

# 处理教育数据：22-23，23-24的CP数据

In [39]:
tasks = [
    {
        'year': '2022-2023',
        'file_path': r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\Dictee_jour_CP_2022_2023.xlsx',
        'date_mapping': date_mapping_2022,
        'output_name': 'final_merged_CP_2022.csv'
    },
    {
        'year': '2023-2024',
        'file_path': r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\DICTEE_CP_2023_2024.xls',
        'date_mapping': date_mapping_2023,
        'output_name': 'final_merged_CP_2023.csv'
    }
]

for task in tasks:
    print(f"   开始处理CP文件。处理年份: {task['year']}")
    print(f"   读取文件: {task['file_path']}")
    df_edu = pd.read_excel(task['file_path'], sheet_name=0)
    
    
    ##公立/私立学校0-1处理
    df_edu = df_edu[df_edu['Statut'] != 'Communautaire']
    df_edu = df_edu.dropna(subset=['Statut'])
    Statut_mapping = {
        'Public': 0,
        'Privé': 1
    }
    df_edu['Statut'] = df_edu['Statut'].replace(Statut_mapping)
    try:
        df_edu['Statut'] = df_edu['Statut'].astype(int)
    except ValueError:
        print("警告：'Statut' 字段转换整数失败，可能存在未处理的非数值/非映射值。")


    ##城市/乡村学校0-1处理
    df_edu = df_edu.dropna(subset=['Milieu_ecole'])
    Milieu_ecole_mapping = {
        'Rural': 0,
        'Urbain': 1
    }
    df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].replace(Milieu_ecole_mapping)
    try:
        df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].astype(int)
    except ValueError:
        print("警告：'Milieu_ecole' 字段转换整数失败，可能存在未处理的非数值/非映射值。")




    ##年级创建哑变量
    new_level_columns = ['NIVEAU2', 'NIVEAU3', 'NIVEAU4', 'NIVEAU5', 'NIVEAU6']
    for col in new_level_columns:
        df_edu[col] = 0
    df_edu['NIVEAU2'] = np.where(df_edu['Clas'] == 'CP2', 1, df_edu['NIVEAU2'])
    df_edu['NIVEAU3'] = np.where(df_edu['Clas'] == 'CE1', 1, df_edu['NIVEAU3'])
    df_edu['NIVEAU4'] = np.where(df_edu['Clas'] == 'CE2', 1, df_edu['NIVEAU4'])
    df_edu['NIVEAU5'] = np.where(df_edu['Clas'] == 'CM1', 1, df_edu['NIVEAU5'])
    df_edu['NIVEAU6'] = np.where(df_edu['Clas'] == 'CM2', 1, df_edu['NIVEAU6'])


    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    ##将男生女生数据单列不同行，并加上性别0-1标签
    cols_triplets = [
        ('Eff_eleve_inscrit', 'Eff_Fille_inscrit', 'Eff_Garc_inscrit', 'Eff_inscrit_Sexe'),
        ('Eff_eleve_present', 'Eff_Fille_present', 'Eff_Garc_present', 'Eff_present_Sexe'),
        ('eff_eleve_absent',  'Eff_fille_absent',  'Eff_Garc_absent',  'Eff_absent_Sexe'),
    
        ('mauvaise_graphie_0_fte',   'mauvaise_graphie_F_0_fte',   'mauvaise_graphie_G_0_fte',   'mauvaise_graphie_0_Sexe'),
        ('mauvaise_graphie_1_5_fte', 'mauvaise_graphie_F_1_5_fte', 'mauvaise_graphie_G_1_5_fte', 'mauvaise_graphie_1_5_Sexe'),
        ('mauvaise_graphie_6_fte',   'mauvaise_graphie_f_6_fte',   'mauvaise_graphie_g_6_fte',   'mauvaise_graphie_6_Sexe'),
    
        ('mots_mal_ortho_0_fte',     'mots_mal_ortho_f_0_fte',     'mots_mal_ortho_g_0_fte',     'mots_mal_ortho_0_Sexe'),
        ('mots_mal_ortho_1_5_fte',   'mots_mal_ortho_f_1_5_fte',   'mots_mal_ortho_G_1_5_fte',   'mots_mal_ortho_1_5_Sexe'),
        ('mots_mal_ortho_6_fte',     'mots_mal_ortho_f_6_fte',     'mots_mal_ortho_g_6_fte',     'mots_mal_ortho_6_Sexe'),
    ]
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@   不同年级处理变的地方

    #构建女生数据 (sexe = 0)
    df_female = df_edu.copy()
    df_female['sexe'] = 0
    rename_dict_f = {triplet[1]: triplet[3] for triplet in cols_triplets} # 将女生列重命名为通用列
    drop_cols_f = [triplet[2] for triplet in cols_triplets]               # 删除男生列
    df_female = df_female.rename(columns=rename_dict_f).drop(columns=drop_cols_f)
    print(f"女生行数：{len(df_female)}")
    #构建男生数据 (sexe = 1)
    df_male = df_edu.copy()
    df_male['sexe'] = 1
    rename_dict_m = {triplet[2]: triplet[3] for triplet in cols_triplets} # 将男生列重命名为通用列
    drop_cols_m = [triplet[1] for triplet in cols_triplets]               # 删除女生列
    df_male = df_male.rename(columns=rename_dict_m).drop(columns=drop_cols_m)
    print(f"男生行数：{len(df_male)}")
    #合并两个表
    df_edu_long = pd.concat([df_female, df_male], ignore_index=True)
    


    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    ##计算优秀率与及格率
    df_edu_long['PassRate']=0.5*((df_edu_long['mauvaise_graphie_0_Sexe']+df_edu_long['mauvaise_graphie_1_5_Sexe'])/df_edu_long['Eff_present_Sexe']\
                              +(df_edu_long['mots_mal_ortho_0_Sexe']+df_edu_long['mots_mal_ortho_1_5_Sexe'])/df_edu_long['Eff_present_Sexe'])
    df_edu_long['ExcellenceRate']=0.5*(df_edu_long['mauvaise_graphie_0_Sexe']/df_edu_long['Eff_present_Sexe']\
                                    +df_edu_long['mots_mal_ortho_0_Sexe']/df_edu_long['Eff_present_Sexe'])
    #除去录入异常数据(没有录入的，或者分值加总大于总值的)
    df_edu_long = df_edu_long[(df_edu_long['PassRate'].notna()) & (df_edu_long['ExcellenceRate'].notna()) \
                            & (df_edu_long['PassRate'] <= 1) & (df_edu_long['ExcellenceRate'] <= 1)]        #验证鲁棒性（参与听写的人数达到一定值）修改的地方,有无：& (df_edu_long['Eff_present_Sexe']>10)
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@   不同年级处理变的地方



    ##根据相关文件，用Clas字段和SESSION字段确定听写发生的日期
    df_edu_long['SESSION'] = df_edu_long['SESSION'].astype(int)
    #使用 map 进行快速元组匹配，将两列组合成索引 -> 在字典中查找 -> 返回值
    df_edu_long['Matched_Data'] = df_edu_long.set_index(['Clas', 'SESSION']).index.map(task['date_mapping'])



    ## 天气数据与教育数据做连接
    #预处理 merged_climate_table：统一连接键，创建一个 lookup 表，无论是 ADM2_FR 还是 ADM2_REF 都能在同一列中找到
    climate_lookup = pd.melt(
        merged_climate_table,
        id_vars=['date', 'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays','rain_hist_vol','Heavy_Rain','Low_Rain',\
                 'Extreme_Fluct_Month','Extreme_Fluct_Intra','ADM1_FR', 'ADM1_PCODE'],
        value_vars=['ADM2_FR', 'ADM2_REF'],
        value_name='Location_Key'
    ).dropna(subset=['Location_Key'])
    #标准化 Climate 表的 Key (转大写 + 去空格)
    climate_lookup['Location_Key'] = climate_lookup['Location_Key'].astype(str).str.upper().str.strip()
    # 去除重复项
    climate_lookup = climate_lookup.drop_duplicates(subset=['date', 'Location_Key'])

    #预处理日期格式
    df_edu_long['Matched_Data'] = pd.to_datetime(df_edu_long['Matched_Data'])
    climate_lookup['date'] = pd.to_datetime(climate_lookup['date'])


    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    df_edu_long['Lib_DREN_Upper'] = df_edu_long['Lib_DREN'].astype(str).str.upper().str.strip().replace({
        r'ABIDJAN\s+\d+': 'ABIDJAN',  # 正则：匹配 ABIDJAN 后接空格和任意数字，替换为 ABIDJAN
        r'BOUAKE\s+\d+': 'BOUAKE',    # 同上
        'SAN-PEDRO': 'SAN PEDRO'
    }, regex=True)
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@   不同年级处理变的地方



    df_edu_long = df_edu_long.sort_values('Matched_Data')
    climate_lookup = climate_lookup.sort_values('date')

    #使用 merge_asof 进行“最近日期”匹配
    #direction='backward': 找小于等于 Matched_Data 的最大 date
    final_merged = pd.merge_asof(
        df_edu_long,
        climate_lookup,
        left_on='Matched_Data',
        right_on='date',
        left_by='Lib_DREN_Upper',    # 左表的地点列
        right_by='Location_Key', # 右表的统一地点列
        direction='backward'
    )

    final_merged = final_merged.drop(columns=['variable','Lib_DREN_Upper'])
    #加入学年列，以便于后继比较学年固定效应
    final_merged['School_Year']=task['year']

    output_path = rf'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\{task["output_name"]}'
    final_merged.to_csv(output_path, index=False, encoding='utf-8-sig')
    #问题：有三分之一的地区没有匹配上，缺少气候数据：气候数据只在100多个ADM2中的34个有数据；教育数据也只涵盖了41个地区；

    print(f"完成! 已保存至: {output_path}")
    print(f"原始行数: {len(df_edu)} -> 展开后: {len(df_edu_long)} -> 合并后: {len(final_merged)}")
    print("-" * 50)
    

   开始处理CP文件。处理年份: 2022-2023
   读取文件: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\Dictee_jour_CP_2022_2023.xlsx


C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\2845143851.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Statut'] = df_edu['Statut'].replace(Statut_mapping)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\2845143851.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].replace(Milieu_ecole_mapping)


女生行数：17465
男生行数：17465
完成! 已保存至: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_2022.csv
原始行数: 17465 -> 展开后: 34879 -> 合并后: 34879
--------------------------------------------------
   开始处理CP文件。处理年份: 2023-2024
   读取文件: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\DICTEE_CP_2023_2024.xls


C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\2845143851.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Statut'] = df_edu['Statut'].replace(Statut_mapping)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\2845143851.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].replace(Milieu_ecole_mapping)


女生行数：10702
男生行数：10702
完成! 已保存至: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_2023.csv
原始行数: 10702 -> 展开后: 21366 -> 合并后: 21366
--------------------------------------------------


# 处理教育数据：22-23，23-24的CE_CM数据

In [40]:
tasks = [
    {
        'year': '2022-2023',
        'file_path': r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\CE_CM_DICTEE_JOUR_2022_2023.xlsx',
        'date_mapping': date_mapping_2022,
        'output_name': 'final_merged_CE_CM_2022.csv'
    },
    {
        'year': '2023-2024',
        'file_path': r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\DICTEE CE_CM_2023_2024.xls',
        'date_mapping': date_mapping_2023,
        'output_name': 'final_merged_CE_CM_2023.csv'
    }
]

for task in tasks:
    print(f"   开始处理CE_CM文件。处理年份: {task['year']}")
    print(f"   读取文件: {task['file_path']}")
    df_edu = pd.read_excel(task['file_path'], sheet_name=0)
    
    
    ##公立/私立学校0-1处理
    df_edu = df_edu[df_edu['Statut'] != 'Communautaire']
    df_edu = df_edu.dropna(subset=['Statut'])
    Statut_mapping = {
        'Public': 0,
        'Privé': 1
    }
    df_edu['Statut'] = df_edu['Statut'].replace(Statut_mapping)
    try:
        df_edu['Statut'] = df_edu['Statut'].astype(int)
    except ValueError:
        print("警告：'Statut' 字段转换整数失败，可能存在未处理的非数值/非映射值。")


    ##城市/乡村学校0-1处理
    df_edu = df_edu.dropna(subset=['Milieu_ecole'])
    Milieu_ecole_mapping = {
        'Rural': 0,
        'Urbain': 1
    }
    df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].replace(Milieu_ecole_mapping)
    try:
        df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].astype(int)
    except ValueError:
        print("警告：'Milieu_ecole' 字段转换整数失败，可能存在未处理的非数值/非映射值。")



    ##年级创建哑变量
    new_level_columns = ['NIVEAU2', 'NIVEAU3', 'NIVEAU4', 'NIVEAU5', 'NIVEAU6']
    for col in new_level_columns:
        df_edu[col] = 0
    df_edu['NIVEAU2'] = np.where(df_edu['Clas'] == 'CP2', 1, df_edu['NIVEAU2'])
    df_edu['NIVEAU3'] = np.where(df_edu['Clas'] == 'CE1', 1, df_edu['NIVEAU3'])
    df_edu['NIVEAU4'] = np.where(df_edu['Clas'] == 'CE2', 1, df_edu['NIVEAU4'])
    df_edu['NIVEAU5'] = np.where(df_edu['Clas'] == 'CM1', 1, df_edu['NIVEAU5'])
    df_edu['NIVEAU6'] = np.where(df_edu['Clas'] == 'CM2', 1, df_edu['NIVEAU6'])


    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    ##将男生女生数据单列不同行，并加上性别0-1标签
    cols_triplets =[
        ('Eff_eleve_inscrit', 'Eff_Fille_inscrit', 'Eff_Garc_inscrit', 'Eff_inscrit_Sexe'),
        ('Eff_eleve_present', 'Eff_Fille_present', 'Eff_Garc_present', 'Eff_present_Sexe'),
        ('eff_eleve_absent',  'Eff_fille_absent',  'Eff_Garc_absent',  'Eff_absent_Sexe'),

        ('FL_Homophones_lexicaux_0_fte', 'FL_Homophones_lexicaux_F_0_fte', 'FL_Homophones_lexicaux_G_0_fte', 'FL_Homophones_lexicaux_0_Sexe'),
        ('FL_Faute_accents_0_fte', 'FL_Faute_accents_F_0_fte', 'FL_Faute_accents_G_0_fte', 'FL_Faute_accents_0_Sexe'),
        ('FL_Omis_insertion_traits_union_0_fte', 'FL_Omis_insertion_traits_union_F_0_fte', 'FL_Omis_insertion_traits_union_G_0_fte', 'FL_Omis_insertion_traits_union_0_Sexe'),
        ('FL_Omis_adjonction_0_fte', 'FL_Omis_adjonction_F_0_fte', 'FL_Omis_adjonction_G_0_fte', 'FL_Omis_adjonction_0_Sexe'),
        ('FL_Confusion_syllabes_0_fte', 'FL_Confusion_syllabes_F_0_fte', 'FL_Confusion_syllabes_G_0_fte', 'FL_Confusion_syllabes_0_Sexe'),
        ('FG_Confusion_homophones_0_fte', 'FG_Confusion_homophones_F_0_fte', 'FG_Confusion_homophones_G_0_fte', 'FG_Confusion_homophones_0_Sexe'),
        ('FG_Ac_adjectif_couleur_0_fte', 'FG_Ac_adjectif_couleur_F_0_fte', 'FG_Ac_adjectif_couleur_G_0_fte', 'FG_Ac_adjectif_couleur_0_Sexe'),
        ('FG_Confusion_participe_infinitif_0_fte', 'FG_Confusion_participe_infinitif_F_0_fte', 'FG_Confusion_participe_infinitif_G_0_fte', 'FG_Confusion_participe_infinitif_0_Sexe'),
        ('FG_Ac_adjectif_couleur_1_5_fte', 'FG_Ac_adjectif_couleur_F_1_5_fte', 'FG_Ac_adjectif_couleur_G_1_5_fte', 'FG_Ac_adjectif_couleur_1_5_Sexe'),
        ('FG_Ac_adjectif_couleur_6_fte', 'FG_Ac_adjectif_couleur_F_6_fte', 'FG_Ac_adjectif_couleur_G_6_fte', 'FG_Ac_adjectif_couleur_6_Sexe'),
        ('FG_Ac_nom_0_fte', 'FG_Ac_nom_F_0_fte', 'FG_Ac_nom_G_0_fte', 'FG_Ac_nom_0_Sexe'),
        ('FG_Ac_adjectif_0_fte', 'FG_Ac_adjectif_F_0_fte', 'FG_Ac_adjectif_G_0_fte', 'FG_Ac_adjectif_0_Sexe'),
        ('FG_Ac_verbe_0_fte', 'FG_Ac_verbe_F_0_fte', 'FG_Ac_verbe_G_0_fte', 'FG_Ac_verbe_0_Sexe'),
        ('FG_Ac_participe_passé_0_fte', 'FG_Ac_participe_passé_F_0_fte', 'FG_Ac_participe_passé_G_0_fte', 'FG_Ac_participe_passé_0_Sexe'),
        ('FG_Ac_nom_compos_0_fte', 'FG_Ac_nom_compos_F_0_fte', 'FG_Ac_nom_compos_G_0_fte', 'FG_Ac_nom_compos_0_Sexe'),
        ('FC_marque_pluriel_verbes_0_fte', 'FC_marque_pluriel_verbes_F_0_fte', 'FC_marque_pluriel_verbes_G_0_fte', 'FC_marque_pluriel_verbes_0_Sexe'),
        ('FC_Confusion_temps_0_fte', 'FC_Confusion_temps_F_0_fte', 'FC_Confusion_temps_G_0_fte', 'FC_Confusion_temps_0_Sexe'),
        ('FL_Confusion_consonnes_voyelles_0_fte', 'FL_Confusion_consonnes_voyelles_F_0_fte', 'FL_Confusion_consonnes_voyelles_G_0_fte', 'FL_Confusion_consonnes_voyelles_0_Sexe'),
        ('FL_Confusion_consonnes_voyelles_1_5_fte', 'FL_Confusion_consonnes_voyelles_F_1_5_fte', 'FL_Confusion_consonnes_voyelles_G_1_5_fte', 'FL_Confusion_consonnes_voyelles_1_5_Sexe'),
        ('FL_Confusion_consonnes_voyelles_6_fte', 'FL_Confusion_consonnes_voyelles_F_6_fte', 'FL_Confusion_consonnes_voyelles_G_6_fte', 'FL_Confusion_consonnes_voyelles_6_Sexe'),
        ('FL_Consonnes_doubles_0_fte', 'FL_Consonnes_doubles_F_0_fte', 'FL_Consonnes_doubles_G_0_fte', 'FL_Consonnes_doubles_0_Sexe'),
        ('FL_Consonnes_doubles_6_fte', 'FL_Consonnes_doubles_F_6_fte', 'FL_Consonnes_doubles_G_6_fte', 'FL_Consonnes_doubles_6_Sexe'),
        ('FL_Consonnes_doubles_1_5_fte', 'FL_Consonnes_doubles_F_1_5_fte', 'FL_Consonnes_doubles_G_1_5_fte', 'FL_Consonnes_doubles_1_5_Sexe'),
        ('FL_Lettres_muettes_6_fte', 'FL_Lettres_muettes_F_6_fte', 'FL_Lettres_muettes_G_6_fte', 'FL_Lettres_muettes_6_Sexe'),
        ('FL_Lettres_muettes_1_5_fte', 'FL_Lettres_muettes_F_1_5_fte', 'FL_Lettres_muettes_G_1_5_fte', 'FL_Lettres_muettes_1_5_Sexe'),
        ('FL_Lettres_muettes_0_fte', 'FL_Lettres_muettes_F_0_fte', 'FL_Lettres_muettes_G_0_fte', 'FL_Lettres_muettes_0_Sexe'),
        ('FL_Homophones_lexicaux_1_5_fte', 'FL_Homophones_lexicaux_F_1_5_fte', 'FL_Homophones_lexicaux_G_1_5_fte', 'FL_Homophones_lexicaux_1_5_Sexe'),
        ('FL_Faute_accents_1_5_fte', 'FL_Faute_accents_F_1_5_fte', 'FL_Faute_accents_G_1_5_fte', 'FL_Faute_accents_1_5_Sexe'),
        ('FL_Omis_insertion_traits_union_1_5_fte', 'FL_Omis_insertion_traits_union_F_1_5_fte', 'FL_Omis_insertion_traits_union_G_1_5_fte', 'FL_Omis_insertion_traits_union_1_5_Sexe'),
        ('FL_Omis_adjonction_1_5_fte', 'FL_Omis_adjonction_F_1_5_fte', 'FL_Omis_adjonction_G_1_5_fte', 'FL_Omis_adjonction_1_5_Sexe'),
        ('FL_Confusion_syllabes_1_5_fte', 'FL_Confusion_syllabes_F_1_5_fte', 'FL_Confusion_syllabes_G_1_5_fte', 'FL_Confusion_syllabes_1_5_Sexe'),
        ('FG_Confusion_homophones_1_5_fte', 'FG_Confusion_homophones_F_1_5_fte', 'FG_Confusion_homophones_G_1_5_fte', 'FG_Confusion_homophones_1_5_Sexe'),
        ('FG_Confusion_participe_infinitif_1_5_fte', 'FG_Confusion_participe_infinitif_F_1_5_fte', 'FG_Confusion_participe_infinitif_G_1_5_fte', 'FG_Confusion_participe_infinitif_1_5_Sexe'),
        ('FG_Ac_nom_1_5_fte', 'FG_Ac_nom_F_1_5_fte', 'FG_Ac_nom_G_1_5_fte', 'FG_Ac_nom_1_5_Sexe'),
        ('FG_Ac_adjectif_1_5_fte', 'FG_Ac_adjectif_F_1_5_fte', 'FG_Ac_adjectif_G_1_5_fte', 'FG_Ac_adjectif_1_5_Sexe'),
        ('FG_Ac_verbe_1_5_fte', 'FG_Ac_verbe_F_1_5_fte', 'FG_Ac_verbe_G_1_5_fte', 'FG_Ac_verbe_1_5_Sexe'),
        ('FG_Ac_participe_passé_1_5_fte', 'FG_Ac_participe_passé_F_1_5_fte', 'FG_Ac_participe_passé_G_1_5_fte', 'FG_Ac_participe_passé_1_5_Sexe'),
        ('FG_Ac_nom_compos_1_5_fte', 'FG_Ac_nom_compos_F_1_5_fte', 'FG_Ac_nom_compos_G_1_5_fte', 'FG_Ac_nom_compos_1_5_Sexe'),
        ('FC_marque_pluriel_verbes_1_5_fte', 'FC_marque_pluriel_verbes_F_1_5_fte', 'FC_marque_pluriel_verbes_G_1_5_fte', 'FC_marque_pluriel_verbes_1_5_Sexe'),
        ('FC_Confusion_temps_1_5_fte', 'FC_Confusion_temps_F_1_5_fte', 'FC_Confusion_temps_G_1_5_fte', 'FC_Confusion_temps_1_5_Sexe'),
        ('FL_Homophones_lexicaux_6_fte', 'FL_Homophones_lexicaux_F_6_fte', 'FL_Homophones_lexicaux_G_6_fte', 'FL_Homophones_lexicaux_6_Sexe'),
        ('FL_Faute_accents_6_fte', 'FL_Faute_accents_F_6_fte', 'FL_Faute_accents_G_6_fte', 'FL_Faute_accents_6_Sexe'),
        ('FL_Omis_insertion_traits_union_6_fte', 'FL_Omis_insertion_traits_union_F_6_fte', 'FL_Omis_insertion_traits_union_G_6_fte', 'FL_Omis_insertion_traits_union_6_Sexe'),
        ('FL_Omis_adjonction_6_fte', 'FL_Omis_adjonction_F_6_fte', 'FL_Omis_adjonction_G_6_fte', 'FL_Omis_adjonction_6_Sexe'),
        ('FL_Confusion_syllabes_6_fte', 'FL_Confusion_syllabes_F_6_fte', 'FL_Confusion_syllabes_G_6_fte', 'FL_Confusion_syllabes_6_Sexe'),
        ('FG_Confusion_homophones_6_fte', 'FG_Confusion_homophones_F_6_fte', 'FG_Confusion_homophones_G_6_fte', 'FG_Confusion_homophones_6_Sexe'),
        ('FG_Confusion_participe_infinitif_6_fte', 'FG_Confusion_participe_infinitif_F_6_fte', 'FG_Confusion_participe_infinitif_G_6_fte', 'FG_Confusion_participe_infinitif_6_Sexe'),
        ('FG_Ac_nom_6_fte', 'FG_Ac_nom_F_6_fte', 'FG_Ac_nom_G_6_fte', 'FG_Ac_nom_6_Sexe'),
        ('FG_Ac_adjectif_6_fte', 'FG_Ac_adjectif_F_6_fte', 'FG_Ac_adjectif_G_6_fte', 'FG_Ac_adjectif_6_Sexe'),
        ('FG_Ac_verbe_6_fte', 'FG_Ac_verbe_F_6_fte', 'FG_Ac_verbe_G_6_fte', 'FG_Ac_verbe_6_Sexe'),
        ('FG_Ac_participe_passé_6_fte', 'FG_Ac_participe_passé_F_6_fte', 'FG_Ac_participe_passé_G_6_fte', 'FG_Ac_participe_passé_6_Sexe'),
        ('FG_Ac_nom_compos_6_fte', 'FG_Ac_nom_compos_F_6_fte', 'FG_Ac_nom_compos_G_6_fte', 'FG_Ac_nom_compos_6_Sexe'),
        ('FC_marque_pluriel_verbes_6_fte', 'FC_marque_pluriel_verbes_F_6_fte', 'FC_marque_pluriel_verbes_G_6_fte', 'FC_marque_pluriel_verbes_6_Sexe'),
        ('FC_Confusion_temps_6_fte', 'FC_Confusion_temps_F_6_fte', 'FC_Confusion_temps_G_6_fte', 'FC_Confusion_temps_6_Sexe'),
    ]
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@   不同年级处理变的地方


    #构建女生数据 (sexe = 0)
    df_female = df_edu.copy()
    df_female['sexe'] = 0
    rename_dict_f = {triplet[1]: triplet[3] for triplet in cols_triplets} # 将女生列重命名为通用列
    drop_cols_f = [triplet[2] for triplet in cols_triplets]               # 删除男生列
    df_female = df_female.rename(columns=rename_dict_f).drop(columns=drop_cols_f)
    print(f"女生行数：{len(df_female)}")
    #构建男生数据 (sexe = 1)
    df_male = df_edu.copy()
    df_male['sexe'] = 1
    rename_dict_m = {triplet[2]: triplet[3] for triplet in cols_triplets} # 将男生列重命名为通用列
    drop_cols_m = [triplet[1] for triplet in cols_triplets]               # 删除女生列
    df_male = df_male.rename(columns=rename_dict_m).drop(columns=drop_cols_m)
    print(f"男生行数：{len(df_male)}")
    #合并两个表
    df_edu_long = pd.concat([df_female, df_male], ignore_index=True)
    


    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    ##计算优秀率与及格率
    elected_columns_Pass = [triplet[3] for triplet in cols_triplets
                   if '_0_' in triplet[3] or '_1_5_' in triplet[3]]
    elected_columns_Excellence = [triplet[3] for triplet in cols_triplets 
                   if '_0_' in triplet[3]]
    
    df_edu_long['PassRate'] = (df_edu_long[elected_columns_Pass].sum(axis=1))/(18*df_edu_long['Eff_present_Sexe'])
    df_edu_long['ExcellenceRate']=(df_edu_long[elected_columns_Excellence].sum(axis=1))/(18*df_edu_long['Eff_present_Sexe'])


    #除去录入异常数据(没有录入的，或者分值加总大于总值的)
    df_edu_long = df_edu_long[(df_edu_long['PassRate'].notna()) & (df_edu_long['ExcellenceRate'].notna()) \
                            & (df_edu_long['PassRate'] <= 1) & (df_edu_long['ExcellenceRate'] <= 1)]   #验证鲁棒性（参与听写的人数达到一定值）修改的地方,有无：& (df_edu_long['Eff_present_Sexe']>10)
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@   不同年级处理变的地方



    ##根据相关文件，用Clas字段和SESSION字段确定听写发生的日期
    df_edu_long['SESSION'] = df_edu_long['SESSION'].astype(int)
    #使用 map 进行快速元组匹配，将两列组合成索引 -> 在字典中查找 -> 返回值
    df_edu_long['Matched_Data'] = df_edu_long.set_index(['Clas', 'SESSION']).index.map(task['date_mapping'])



    ## 天气数据与教育数据做连接
    #预处理 merged_climate_table：统一连接键，创建一个 lookup 表，无论是 ADM2_FR 还是 ADM2_REF 都能在同一列中找到
    climate_lookup = pd.melt(
        merged_climate_table, 
        id_vars=['date', 'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays','rain_hist_vol','Heavy_Rain','Low_Rain',\
                 'Extreme_Fluct_Month','Extreme_Fluct_Intra','ADM1_FR', 'ADM1_PCODE'],
        value_vars=['ADM2_FR', 'ADM2_REF'],
        value_name='Location_Key'
    ).dropna(subset=['Location_Key'])
    #标准化 Climate 表的 Key (转大写 + 去空格)
    climate_lookup['Location_Key'] = climate_lookup['Location_Key'].astype(str).str.upper().str.strip()
    # 去除重复项
    climate_lookup = climate_lookup.drop_duplicates(subset=['date', 'Location_Key'])

    #预处理日期格式
    df_edu_long['Matched_Data'] = pd.to_datetime(df_edu_long['Matched_Data'])
    climate_lookup['date'] = pd.to_datetime(climate_lookup['date'])


    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    df_edu_long['Lib_DREN_Upper'] = df_edu_long['Lib_DREN'].astype(str).str.upper().str.strip().replace({
        r'ABIDJAN\s+\d+': 'ABIDJAN',  # 正则：匹配 ABIDJAN 后接空格和任意数字，替换为 ABIDJAN
        r'BOUAKE\s+\d+': 'BOUAKE',    # 同上
        'SAN-PEDRO': 'SAN PEDRO'
    }, regex=True)
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@   不同年级处理变的地方



    df_edu_long = df_edu_long.sort_values('Matched_Data')
    climate_lookup = climate_lookup.sort_values('date')

    #使用 merge_asof 进行“最近日期”匹配
    #direction='backward': 找小于等于 Matched_Data 的最大 date
    final_merged = pd.merge_asof(
        df_edu_long,
        climate_lookup,
        left_on='Matched_Data',
        right_on='date',
        left_by='Lib_DREN_Upper',    # 左表的地点列
        right_by='Location_Key', # 右表的统一地点列
        direction='backward'
    )

    final_merged = final_merged.drop(columns=['variable','Lib_DREN_Upper'])
    #加入学年列，以便于后继比较学年固定效应
    final_merged['School_Year']=task['year']

    output_path = rf'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\{task["output_name"]}'
    final_merged.to_csv(output_path, index=False, encoding='utf-8-sig')
    #问题：有三分之一的地区没有匹配上，缺少气候数据：气候数据只在100多个ADM2中的34个有数据；教育数据也只涵盖了41个地区；

    print(f"完成! 已保存至: {output_path}")
    print(f"原始行数: {len(df_edu)} -> 展开后: {len(df_edu_long)} -> 合并后: {len(final_merged)}")
    print("-" * 50)
    

   开始处理CE_CM文件。处理年份: 2022-2023
   读取文件: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\CE_CM_DICTEE_JOUR_2022_2023.xlsx


C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1466833235.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Statut'] = df_edu['Statut'].replace(Statut_mapping)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1466833235.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].replace(Milieu_ecole_mapping)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1466833235.py:152: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert

女生行数：15032
男生行数：15032


C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1466833235.py:166: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_edu_long['Matched_Data'] = df_edu_long.set_index(['Clas', 'SESSION']).index.map(task['date_mapping'])
C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1466833235.py:190: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_edu_long['Lib_DREN_Upper'] = df_edu_long['Lib_DREN'].astype(str).str.upper().str.strip().replace({


完成! 已保存至: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CE_CM_2022.csv
原始行数: 15032 -> 展开后: 30026 -> 合并后: 30026
--------------------------------------------------
   开始处理CE_CM文件。处理年份: 2023-2024
   读取文件: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\DICTEE CE_CM_2023_2024.xls


C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1466833235.py:29: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Statut'] = df_edu['Statut'].replace(Statut_mapping)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1466833235.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].replace(Milieu_ecole_mapping)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1466833235.py:152: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert

女生行数：1552
男生行数：1552
完成! 已保存至: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CE_CM_2023.csv
原始行数: 1552 -> 展开后: 3095 -> 合并后: 3095
--------------------------------------------------


In [41]:
# #将2024-2025的所有文件合并成一张总表
# import os
# files = [
#     r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\GRAMMAIRE_CE_CM_2024_2025_20250723_113109.xlsx',
#     r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\LEXICAL_CE_CM_2024_2025_20250723_112836.xlsx',
#     r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\CONJUGAISON_CE_CM_2024_2025_20250723_113311.xlsx',
#     r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\LEXICAL_CP_2024_2025_20250723_112750.xlsx'
# ]
# dataframes = [pd.read_excel(f) for f in files]
# merged_data = pd.concat(dataframes, ignore_index=True)
# output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\CP_CE_CM_2024_2025.xlsx'
# merged_data.to_excel(output_path, index=False)

# print(f"合并完成！文件已保存为: {output_path}")
# print(f"总数据量: {len(merged_data)} 行 × {len(merged_data.columns)} 列")

# 处理教育数据：24-25的CP, CE, CM数据

In [42]:
tasks = [
    {
        'year': '2024-2025',
        'file_path': r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\CP_CE_CM_2024_2025.xlsx',
        'date_mapping': date_mapping_2024,
        'output_name': 'final_merged_CP_CE_CM_2024.csv'
    }
]

for task in tasks:
    print(f"   开始处理CE,CM,CP文件。处理年份: {task['year']}")
    print(f"   读取文件: {task['file_path']}")
    df_edu = pd.read_excel(task['file_path'], sheet_name=0)

    #只保留录入数据了的行
    df_edu = df_edu[df_edu['TRAITER'] == 1]

    #重新命名字段，使其与之前年份的相同字段名字一致
    df_edu = df_edu.rename(columns={'STATUT': 'Statut'})
    df_edu = df_edu.rename(columns={'MILIEU': 'Milieu_ecole'})
    df_edu = df_edu.rename(columns={'NIVEAU': 'Clas'})
    df_edu = df_edu.rename(columns={'DRENA': 'Lib_DREN'})

    df_edu = df_edu.dropna(subset=['Statut'])
    df_edu = df_edu.dropna(subset=['Milieu_ecole'])

    #合并同一学校、同一日期的不同错误类型数据
    df_edu = df_edu.drop(columns=['TYPE DE FAUTE', 'TRAITER'], errors='ignore')
    group_cols = ['Lib_DREN', 'IEPP', 'ECOLE', 'Statut', 'Milieu_ecole', 'SESSION', 'Clas']
    sum_cols = [
        'PRESENT-TOTAL', 'PRESENT-FILLES', 'PRESENT-GRACONS', 
        'TOTAL (0 FAUTE)', 'FILLES (0 FAUTE)', 'GARÇONS (0 FAUTE)',
        'TOTAL (1-5 FAUTE)', 'FILLES (1-5 FAUTE)', 'GARÇONS (1-5 FAUTE)',
        'TOTAL (6+ FAUTES)', 'FILLES (6+ FAUTES)', 'GARÇONS (6+ FAUTES)'
    ]

    df_edu = df_edu.groupby(group_cols)[sum_cols].sum().reset_index()

    # output_path = rf'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\{task["output_name"]}'
    # df_edu.to_csv(output_path, index=False, encoding='utf-8-sig')



    ##公立/私立学校0-1处理
    #df_edu = df_edu[df_edu['Statut'] != 'Communautaire']
    Statut_mapping = {
        'Public': 0,
        'Privé': 1
    }
    df_edu['Statut'] = df_edu['Statut'].replace(Statut_mapping)
    try:
        df_edu['Statut'] = df_edu['Statut'].astype(int)
    except ValueError:
        print("警告：'Statut' 字段转换整数失败，可能存在未处理的非数值/非映射值。")


    ##城市/乡村学校0-1处理
    Milieu_ecole_mapping = {
        'Rural': 0,
        'Urbain': 1
    }
    df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].replace(Milieu_ecole_mapping)
    try:
        df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].astype(int)
    except ValueError:
        print("警告：'Milieu_ecole' 字段转换整数失败，可能存在未处理的非数值/非映射值。")



    ##年级创建哑变量
    new_level_columns = ['NIVEAU2', 'NIVEAU3', 'NIVEAU4', 'NIVEAU5', 'NIVEAU6']
    for col in new_level_columns:
        df_edu[col] = 0
    df_edu['NIVEAU2'] = np.where(df_edu['Clas'] == 'CP2', 1, df_edu['NIVEAU2'])
    df_edu['NIVEAU3'] = np.where(df_edu['Clas'] == 'CE1', 1, df_edu['NIVEAU3'])
    df_edu['NIVEAU4'] = np.where(df_edu['Clas'] == 'CE2', 1, df_edu['NIVEAU4'])
    df_edu['NIVEAU5'] = np.where(df_edu['Clas'] == 'CM1', 1, df_edu['NIVEAU5'])
    df_edu['NIVEAU6'] = np.where(df_edu['Clas'] == 'CM2', 1, df_edu['NIVEAU6'])


    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    ##将男生女生数据单列不同行，并加上性别0-1标签
    cols_triplets =[
        ('PRESENT-TOTAL', 'PRESENT-FILLES', 'PRESENT-GRACONS', 'PRESENT-SEXE'),

        ('TOTAL (0 FAUTE)', 'FILLES (0 FAUTE)', 'GARÇONS (0 FAUTE)', 'SEXE (0 FAUTE)'),
        ('TOTAL (1-5 FAUTE)', 'FILLES (1-5 FAUTE)', 'GARÇONS (1-5 FAUTE)', 'SEXE (1-5 FAUTE)'),
        ('TOTAL (6+ FAUTES)', 'FILLES (6+ FAUTES)', 'GARÇONS (6+ FAUTES)', 'SEXE (6+ FAUTES)')
    ]
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@   不同年级处理变的地方


    #构建女生数据 (sexe = 0)
    df_female = df_edu.copy()
    df_female['sexe'] = 0
    rename_dict_f = {triplet[1]: triplet[3] for triplet in cols_triplets} # 将女生列重命名为通用列
    drop_cols_f = [triplet[2] for triplet in cols_triplets]               # 删除男生列
    df_female = df_female.rename(columns=rename_dict_f).drop(columns=drop_cols_f)
    print(f"女生行数：{len(df_female)}")
    #构建男生数据 (sexe = 1)
    df_male = df_edu.copy()
    df_male['sexe'] = 1
    rename_dict_m = {triplet[2]: triplet[3] for triplet in cols_triplets} # 将男生列重命名为通用列
    drop_cols_m = [triplet[1] for triplet in cols_triplets]               # 删除女生列
    df_male = df_male.rename(columns=rename_dict_m).drop(columns=drop_cols_m)
    print(f"男生行数：{len(df_male)}")
    #合并两个表
    df_edu_long = pd.concat([df_female, df_male], ignore_index=True)
    

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    ##计算优秀率与及格率
    df_edu_long['PassRate']=(df_edu_long['SEXE (0 FAUTE)']+df_edu_long['SEXE (1-5 FAUTE)'])/df_edu_long['PRESENT-SEXE']
    df_edu_long['ExcellenceRate']=df_edu_long['SEXE (0 FAUTE)']/df_edu_long['PRESENT-SEXE']
    #除去录入异常数据(没有录入的，或者分值加总大于总值的)
    df_edu_long = df_edu_long[(df_edu_long['PassRate'].notna()) & (df_edu_long['ExcellenceRate'].notna()) \
                            & (df_edu_long['PassRate'] <= 1) & (df_edu_long['ExcellenceRate'] <= 1)]   #验证鲁棒性（参与听写的人数达到一定值）修改的地方,是否有： & (df_edu_long['PRESENT-SEXE']>10)
    
    # output_path = rf'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\{task["output_name"]}'
    # df_edu_long.to_csv(output_path, index=False, encoding='utf-8-sig')
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@   不同年级处理变的地方



    ##根据相关文件，用Clas字段和SESSION字段确定听写发生的日期
    # df_edu_long['SESSION'] = df_edu_long['SESSION'].astype(int)
    #使用 map 进行快速元组匹配，将两列组合成索引 -> 在字典中查找 -> 返回值
    df_edu_long['Matched_Data'] = df_edu_long.set_index(['Clas', 'SESSION']).index.map(task['date_mapping'])



    ## 天气数据与教育数据做连接
    #预处理 merged_climate_table：统一连接键，创建一个 lookup 表，无论是 ADM2_FR 还是 ADM2_REF 都能在同一列中找到
    climate_lookup = pd.melt(
        merged_climate_table, 
        id_vars=['date', 'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays','rain_hist_vol','Heavy_Rain','Low_Rain',\
                 'Extreme_Fluct_Month','Extreme_Fluct_Intra','ADM1_FR', 'ADM1_PCODE'],
        value_vars=['ADM2_FR', 'ADM2_REF'],
        value_name='Location_Key'
    ).dropna(subset=['Location_Key'])
    #标准化 Climate 表的 Key (转大写 + 去空格)
    climate_lookup['Location_Key'] = climate_lookup['Location_Key'].astype(str).str.upper().str.strip()
    # 去除重复项
    climate_lookup = climate_lookup.drop_duplicates(subset=['date', 'Location_Key'])

    #预处理日期格式
    df_edu_long['Matched_Data'] = pd.to_datetime(df_edu_long['Matched_Data'])
    climate_lookup['date'] = pd.to_datetime(climate_lookup['date'])


    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    #去掉Lib_DREN字段的前缀'DRENA '；
    df_edu_long['Lib_DREN'] = df_edu_long['Lib_DREN'].str.replace('DRENA ', '', regex=False)

    df_edu_long['Lib_DREN_Upper'] = df_edu_long['Lib_DREN'].astype(str).str.upper().str.strip().replace({
        r'ABIDJAN\s+\d+': 'ABIDJAN',  # 正则：匹配 ABIDJAN 后接空格和任意数字，替换为 ABIDJAN
        r'BOUAKE\s+\d+': 'BOUAKE',    # 同上
        'SAN-PEDRO': 'SAN PEDRO'
    }, regex=True)
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@   不同年级处理变的地方



    df_edu_long = df_edu_long.sort_values('Matched_Data')
    climate_lookup = climate_lookup.sort_values('date')

    #使用 merge_asof 进行“最近日期”匹配
    #direction='backward': 找小于等于 Matched_Data 的最大 date
    final_merged = pd.merge_asof(
        df_edu_long,
        climate_lookup,
        left_on='Matched_Data',
        right_on='date',
        left_by='Lib_DREN_Upper',    # 左表的地点列
        right_by='Location_Key', # 右表的统一地点列
        direction='backward'
    )

    final_merged = final_merged.drop(columns=['variable','Lib_DREN_Upper'])
    #加入学年列，以便于后继比较学年固定效应
    final_merged['School_Year']=task['year']

    output_path = rf'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\{task["output_name"]}'
    final_merged.to_csv(output_path, index=False, encoding='utf-8-sig')
    #问题：有三分之一的地区没有匹配上，缺少气候数据：气候数据只在100多个ADM2中的34个有数据；教育数据也只涵盖了41个地区；

    print(f"完成! 已保存至: {output_path}")
    print(f"原始行数: {len(df_edu)} -> 展开后: {len(df_edu_long)} -> 合并后: {len(final_merged)}")
    print("-" * 50)

   开始处理CE,CM,CP文件。处理年份: 2024-2025
   读取文件: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Source Data for Python Analyse\CP_CE_CM_2024_2025.xlsx


C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1422062925.py:50: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Statut'] = df_edu['Statut'].replace(Statut_mapping)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1422062925.py:62: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_edu['Milieu_ecole'] = df_edu['Milieu_ecole'].replace(Milieu_ecole_mapping)


女生行数：10994
男生行数：10994
完成! 已保存至: C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_CE_CM_2024.csv
原始行数: 10994 -> 展开后: 21807 -> 合并后: 21807
--------------------------------------------------


# 选取要做回归的变量，将所有中间文件合并成一个最终数据文件

In [ ]:

columns = ['Lib_DREN','Ecole', 'Statut', 'Milieu_ecole', 'Clas',\
           'NIVEAU2', 'NIVEAU3', 'NIVEAU4', 'NIVEAU5', 'NIVEAU6',\
           'sexe', 'PassRate', 'ExcellenceRate', 'date', \
           'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays','rain_hist_vol','Heavy_Rain','Low_Rain',\
           'Extreme_Fluct_Month','Extreme_Fluct_Intra',\
           'ADM1_FR', 'ADM1_PCODE','School_Year']

df_CP_CE_CM_2024 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_CE_CM_2024.csv')
df_CP_CE_CM_2024 = df_CP_CE_CM_2024.rename(columns={'ECOLE': 'Ecole'})
df_CP_CE_CM_2024 = df_CP_CE_CM_2024.dropna(subset=['r1h'])
df_CP_CE_CM_2024 = df_CP_CE_CM_2024[columns]


df_CE_CM_2023 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CE_CM_2023.csv')
df_CE_CM_2023 = df_CE_CM_2023.dropna(subset=['r1h'])
df_CE_CM_2023 = df_CE_CM_2023[columns]


df_CE_CM_2022 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CE_CM_2022.csv')
df_CE_CM_2022 = df_CE_CM_2022.dropna(subset=['r1h'])
df_CE_CM_2022 = df_CE_CM_2022[columns]


df_CP_2023 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_2023.csv')
df_CP_2023 = df_CP_2023.dropna(subset=['r1h'])
df_CP_2023 = df_CP_2023[columns]


df_CP_2022 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_2022.csv')
df_CP_2022 = df_CP_2022.dropna(subset=['r1h'])
df_CP_2022 = df_CP_2022[columns]

all_data_final = pd.concat([df_CP_CE_CM_2024,df_CE_CM_2023,df_CE_CM_2022,df_CP_2023,df_CP_2022], ignore_index=True)
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final.csv'
all_data_final.to_csv(output_path, index=False)
#all_data_final表check过，各个学校的气候数据列对得上没问题

# 给教育部门分析数据要用到的文件，与thesis无关

In [ ]:

# columns = ['Lib_DREN','Ecole', 'Statut', 'Milieu_ecole', 'Clas',\
#            'NIVEAU2', 'NIVEAU3', 'NIVEAU4', 'NIVEAU5', 'NIVEAU6',\
#            'sexe', 'PassRate', 'ExcellenceRate','Matched_Data', 'date', \
#            'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays','rain_hist_vol','Heavy_Rain','Low_Rain',\
#            'Extreme_Fluct_Month','Extreme_Fluct_Intra',\
#            'ADM1_FR', 'ADM1_PCODE','School_Year']

# df_CP_CE_CM_2024 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_CE_CM_2024.csv')
# df_CP_CE_CM_2024 = df_CP_CE_CM_2024.rename(columns={'ECOLE': 'Ecole'})
# # df_CP_CE_CM_2024 = df_CP_CE_CM_2024.dropna(subset=['r1h'])
# df_CP_CE_CM_2024 = df_CP_CE_CM_2024[columns]


# df_CE_CM_2023 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CE_CM_2023.csv')
# # df_CE_CM_2023 = df_CE_CM_2023.dropna(subset=['r1h'])
# df_CE_CM_2023 = df_CE_CM_2023[columns]


# df_CE_CM_2022 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CE_CM_2022.csv')
# # df_CE_CM_2022 = df_CE_CM_2022.dropna(subset=['r1h'])
# df_CE_CM_2022 = df_CE_CM_2022[columns]


# df_CP_2023 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_2023.csv')
# # df_CP_2023 = df_CP_2023.dropna(subset=['r1h'])
# df_CP_2023 = df_CP_2023[columns]


# df_CP_2022 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_2022.csv')
# # df_CP_2022 = df_CP_2022.dropna(subset=['r1h'])
# df_CP_2022 = df_CP_2022[columns]

# all_data_final = pd.concat([df_CP_CE_CM_2024,df_CE_CM_2023,df_CE_CM_2022,df_CP_2023,df_CP_2022], ignore_index=True)
# output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_for_education_sector.csv'
# all_data_final.to_csv(output_path, index=False)
# #all_data_final表check过，各个学校的气候数据列对得上没问题

C:\Users\ASUS\AppData\Local\Temp\ipykernel_38536\1080772550.py:29: DtypeWarning: Columns (0,7,13,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_CP_2022 = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\final_merged_CP_2022.csv')
